In [1]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook', 'grid'])

In [2]:
N = 250

t, g = sp.symbols('t g')
m1, m2, m3 = sp.symbols('m1 m2 m3')
L1, L2, L3 = sp.symbols('L1 L2 L3')

In [3]:
the1, the2, the3 = sp.symbols(r'\theta_1, \theta_2, \theta_3',cls=sp.Function)

the1 = the1(t)
the2 = the2(t)
the3 = the3(t)

In [4]:
the1_d = sp.diff(the1, t)
the2_d = sp.diff(the2, t)
the3_d = sp.diff(the3, t)
the1_dd = sp.diff(the1_d, t)
the2_dd = sp.diff(the2_d, t)
the3_dd = sp.diff(the3_d, t)

In [7]:
x1 = L1 * sp.sin(the1)
y1 = -L1 * sp.cos(the1)
x2 = L1 * sp.sin(the1) + L2 * sp.sin(the2)
y2 = -L1 * sp.cos(the1) - L2 * sp.cos(the2)
x3 = L1 * sp.sin(the1) + L2 * sp.sin(the2)+ L3 * sp.sin(the3)
y3 = -L1 * sp.cos(the1) - L2 * sp.cos(the2)- L3 * sp.cos(the3)

In [8]:
# Kinetic Energy Term
T1 = 0.5 * m1 * (sp.diff(x1,t)**2+sp.diff(y1,t)**2)
T2 = 0.5 * m2 * (sp.diff(x2,t)**2+sp.diff(y2,t)**2)
T3 = 0.5 * m3 * (sp.diff(x3,t)**2+sp.diff(y3,t)**2)
T = T1 + T2 + T3

# Potential Energy Term
U1 = m1 * g * y1
U2 = m2 * g * y2
U3 = m3 * g * y3
U = U1 + U2 + U3

# Lagrangian
L = T - U

In [9]:
LE1 = sp.diff(L,the1) - sp.diff(sp.diff(L,the1_d),t).simplify()
LE2 = sp.diff(L,the2) - sp.diff(sp.diff(L,the2_d),t).simplify()
LE3 = sp.diff(L,the3) - sp.diff(sp.diff(L,the3_d),t).simplify()

In [10]:
sols = sp.solve([LE1, LE2, LE3], (the1_dd, the2_dd, the3_dd), simplify = False, rational=False)

In [11]:
dw_1dt_f = sp.lambdify((t,g,m1,m2, m3,L1,L2,L3,the1,the2,the3,the1_d,the2_d,the3_d), sols[the1_dd])
dw_2dt_f = sp.lambdify((t,g,m1,m2, m3,L1,L2,L3,the1,the2,the3,the1_d,the2_d,the3_d), sols[the2_dd])
dw_3dt_f = sp.lambdify((t,g,m1,m2, m3,L1,L2,L3,the1,the2,the3,the1_d,the2_d,the3_d), sols[the3_dd])
dthe1dt_f = sp.lambdify(the1_d,the1_d)
dthe2dt_f = sp.lambdify(the2_d,the2_d)
dthe3dt_f = sp.lambdify(the3_d,the3_d)

In [12]:
def dSdt(S, t ,g, m1, m2, m3, L1, L2, L3):
    the1, w1, the2, w2, the3, w3 = S
    return [
        dthe1dt_f(w1),
        dw_1dt_f(t, g, m1, m2, m3, L1, L2, L3, the1, the2, the3, w1, w2, w3),
        dthe2dt_f(w2),
        dw_2dt_f(t, g, m1, m2, m3, L1, L2, L3, the1, the2, the3, w1, w2, w3),
        dthe3dt_f(w3),
        dw_3dt_f(t, g, m1, m2, m3, L1, L2, L3, the1, the2, the3, w1, w2, w3)
    ]

In [ ]:
t = np.linspace(0, 5, N)
g = 9.81
m1 = 2
m2 = 1
m3 = 1
L1 = 2
L2 = 1
L3 = 1
ans = sc.integrate.odeint(dSdt, y0=[-np.pi/2, 0, 1, 0, 2, 0], t=t, args=(g,m1,m2,m3,L1,L2,L3))

In [51]:
the1 = ans.T[0]
the2 = ans.T[2]
the3 = ans.T[4]
plt.plot(t,the1)
plt.plot(t,the2)
plt.plot(t,the3)

In [52]:
def get_x1y1x2y2x3y3(t,the1,the2,the3,L1,L2,L3):
    return (
        L1 * np.sin(the1),
        -L1 * np.cos(the1),
        L1 * np.sin(the1) + L2 * np.sin(the2),
        -L1 * np.cos(the1) - L2 * np.cos(the2),
        L1 * np.sin(the1) + L2 * np.sin(the2) + L3 * np.sin(the3),
        -L1 * np.cos(the1) - L2 * np.cos(the2) - L3 * np.cos(the3)
    )

x1, y1, x2, y2, x3, y3 = get_x1y1x2y2x3y3(t, ans.T[0], ans.T[2], ans.T[4], L1, L2, L3)

In [53]:
def animate(i):
    ln1.set_data([0, x1[i], x2[i], x3[i]], [0, y1[i], y2[i], y3[i]])

In [ ]:
from matplotlib import pyplot as plt, animation

fig, ax = plt.subplots()
ax.set_facecolor('k')
ax.set(xlim=(-6, 6), ylim=(-6, 6))
ln1, = plt.plot([], [], 'ro--', markersize=6)

ani = animation.FuncAnimation(fig, animate, frames = N-1, interval = 50)
#ani.save(filename="/Users/hasan/Python Animations/Triple Pendulum.gif", writer="pillow")
#plt.show(ani)